# FedX-TinyIDS: Budget-Constrained Explainable Federated TinyML IDS

Implements the baseline and all experiments defined in `FedTinyIDS.tex`, with real (measured or deterministically-derived) substitutes for the previously mocked components: INT8 quantization, endpoint memory/latency profiling, BCES utility signals, and SHAP/LIME explanation cost.

## Imports

In [ ]:
import io
import json
import time
import warnings
from collections import defaultdict, deque
from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

In [ ]:
from scipy.stats import wilcoxon
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore")

### Optional dependencies (Opacus / SHAP / LIME)

In [ ]:
try:
    from opacus import PrivacyEngine
    from opacus.accountants import RDPAccountant
    OPACUS_AVAILABLE = True
except ImportError:
    PrivacyEngine = RDPAccountant = None
    OPACUS_AVAILABLE = False

In [ ]:
try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    shap = None
    SHAP_AVAILABLE = False

In [ ]:
try:
    from lime.lime_tabular import LimeTabularExplainer
    LIME_AVAILABLE = True
except ImportError:
    LimeTabularExplainer = None
    LIME_AVAILABLE = False

In [ ]:
if not (SHAP_AVAILABLE and LIME_AVAILABLE):
    print(
        "[WARN] shap/lime not fully available (shap_available="
        f"{SHAP_AVAILABLE}, lime_available={LIME_AVAILABLE}). "
        "Run `pip install shap lime` to get real explanation-latency "
        "measurements; without them, the XAI overhead table cannot be "
        "populated honestly and BCES_USE_REAL_XAI must be left False."
    )

## Pipeline execution flags

In [ ]:
RUN_CENTRALIZED_FP32 = True
RUN_FEDAVG_INT8 = True
RUN_FEDPROX_FP32 = True
RUN_FEDX_TINYIDS = True

RUN_ABLATION_TRUST = True
RUN_ABLATION_DP = True
RUN_ABLATION_BCES = True

In [ ]:
# If True and real shap/lime are installed, BCES/XAI experiments call the
# real explainers and time them. If False, XAI experiments are skipped
# entirely (no fabricated numbers are produced as a fallback).
BCES_USE_REAL_XAI = SHAP_AVAILABLE and LIME_AVAILABLE

# Synthetic data is a debugging aid ONLY. It must be turned on explicitly and
# is loudly flagged in every output artifact; it must never be the silent
# default behind a missing dataset path.
ALLOW_SYNTHETIC_DEBUG_DATA = False

## Dataset & federated hyperparameters

In [ ]:
SELECTED_DATASETS = ["ton_iot"]  # Support: 'ton_iot', 'ciciot2023', 'edge_iiot'
EPOCHS = 50
ROUNDS = 25
LOCAL_EPOCHS = 10
N_CLIENTS = 10
NON_IID_ALPHA = 0.1

In [ ]:
BATCH = 256
LR = 0.002
FEDPROX_MU = 0.01

SEEDS = [42, 43, 44, 45, 46]  # Section 5.6 requires >= 5 seeds
MAX_SAMPLES = None

In [ ]:
DP_CLIPPING_NORM = 1.0
DP_NOISE_MULTIPLIER = 1.0
DP_SIGMA_SWEEP = [0.5, 1.0, 1.5]

In [ ]:
BCES_BUDGET_MS = 200.0
BCES_NOVELTY_BUFFER_SIZE = 200
# Per-macro-class severity prior used for A(x); documented assumption, not a
# measured quantity. 0=Normal,1=Volumetric,2=Recon,3=Spoofing/MITM,4=Exploit/Web
SEVERITY_TABLE = {0: 0.0, 1: 0.8, 2: 0.4, 3: 0.7, 4: 1.0}

In [ ]:
# Port buckets treated as higher-value targets for the device-criticality
# proxy R(x); documented assumption (e.g. RDP/SMB/SQL endpoints are common
# lateral-movement / high-value targets in IIoT environments).
HIGH_CRITICALITY_PORT_BUCKETS = {"rdp", "smb", "sql", "mysql"}

In [ ]:
ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
DATASETS_ROOT = (ROOT / "../../dataset").resolve()
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
MACRO_CLASSES = ["Normal", "Volumetric", "Reconnaissance", "Spoofing/MITM", "Exploitation/Web"]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training device: {DEVICE}")
BENCH_DEVICE = torch.device("cpu")  # latency/memory benchmarks always run on CPU as an MCU proxy

In [ ]:
def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

## Data loading and harmonization

In [ ]:
_DROP_COLS = {
    "src_ip", "dst_ip", "dns_query", "ssl_subject", "ssl_issuer",
    "http_uri", "http_user_agent", "weird_name",
}
_LOW_CARD_CATS = [
    "proto", "service", "conn_state", "http_method", "http_version",
    "http_orig_mime_types", "http_resp_mime_types", "ssl_version", "ssl_cipher",
    "ssl_resumed", "ssl_established", "dns_AA", "dns_RD", "dns_RA", "dns_rejected",
    "weird_addl", "weird_notice",
]
_TOPN_PER_CAT = 8

In [ ]:
WELL_KNOWN_PORTS = {
    80: "http", 443: "https", 53: "dns", 22: "ssh",
    21: "ftp", 23: "telnet", 25: "smtp", 3389: "rdp",
    445: "smb", 1433: "sql", 3306: "mysql", 8080: "http_alt",
}

In [ ]:
_LABEL_MAPS = {
    "ton_iot": {
        "normal": 0, "dos": 1, "ddos": 1, "scanning": 2, "mitm": 3,
        "injection": 4, "xss": 4, "password": 4, "ransomware": 4, "backdoor": 4,
    },
    "ciciot2023": {
        "benign": 0, "ddos": 1, "dos": 1, "mirai": 1, "spoofing": 3,
        "recon": 2, "web": 4, "bruteforce": 4,
    },
    "edge_iiot": {
        "normal": 0, "ddos": 1, "port_scanning": 2, "os_fingerprinting": 2,
        "vulnerability_scanner": 2, "mitm": 3, "sql_injection": 4, "xss": 4,
        "ransomware": 4, "backdoor": 4, "password": 4,
    },
}

In [ ]:
def map_labels(df, dataset_name):
    if dataset_name not in _LABEL_MAPS:
        raise ValueError(f"Unknown dataset {dataset_name}")
    s = df["__label__"].astype(str).str.strip().str.lower()
    return s.map(_LABEL_MAPS[dataset_name]).fillna(0).astype(np.int64)

In [ ]:
def _bucketize_ports(df):
    """Adds one-hot port-bucket columns AND returns the raw bucket label per
    row (kept out-of-band) so BCES can use it later as a device-criticality
    proxy without re-deriving it from one-hot columns."""
    bucket_labels = {}
    for col in ("src_port", "dst_port"):
        if col not in df.columns:
            continue
        s = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)
        bucket = s.map(WELL_KNOWN_PORTS)
        bucket = bucket.where(s.isin(WELL_KNOWN_PORTS.keys()), np.where(s >= 1024, "high", "low"))
        bucket_labels[col] = bucket
        dummies = pd.get_dummies(bucket, prefix=f"{col}_bkt", dtype=np.float32)
        df = pd.concat([df, dummies], axis=1)
    return df, bucket_labels

In [ ]:
def _encode_categoricals(df, cat_cols, top_n=_TOPN_PER_CAT):
    one_hot_frames = []
    for col in cat_cols:
        if col not in df.columns:
            continue
        s = df[col].astype(str).fillna("-")
        top = s.value_counts().nlargest(top_n).index.tolist()
        s = s.where(s.isin(top), "_other_")
        dummies = pd.get_dummies(s, prefix=col, dtype=np.float32)
        one_hot_frames.append(dummies)
        df = df.drop(columns=[col])
    if one_hot_frames:
        df = pd.concat([df] + one_hot_frames, axis=1)
    return df

In [ ]:
def _resolve_csv_files(dataset_name):
    path_map = {"ton_iot": "ton_iot", "ciciot2023": "CICIOT23", "edge_iiot": "Edge_iiot"}
    if dataset_name not in path_map:
        raise ValueError(f"Unknown dataset {dataset_name}")
    path = DATASETS_ROOT / path_map[dataset_name]
    csv_files = sorted(path.rglob("*.csv")) if path.exists() else []
    return path, csv_files

In [ ]:
def _synthetic_debug_data(dataset_name, path, max_rows):
    msg = (
        f"No CSV files found for dataset '{dataset_name}' at expected path "
        f"'{path}'. Place the harmonized CSV exports there before running, "
        f"or set ALLOW_SYNTHETIC_DEBUG_DATA=True at the top of this script "
        f"if you explicitly want a synthetic-data smoke test."
    )
    if not ALLOW_SYNTHETIC_DEBUG_DATA:
        raise FileNotFoundError(msg)
    for _ in range(3):
        print(f"[SYNTHETIC DATA WARNING] {msg} Returning RANDOM NOISE data ")
    n = max_rows if max_rows else 5000
    X = np.random.rand(n, 39).astype(np.float32)
    y = np.random.randint(0, 5, size=(n,))
    meta = pd.DataFrame({"dst_port_bucket": ["low"] * n})
    return X, y, meta

In [ ]:
def _read_csvs_and_extract_label(csv_files, dataset_name):
    dfs = [pd.read_csv(f, low_memory=False) for f in csv_files]
    full = pd.concat(dfs, ignore_index=True)
    full.columns = full.columns.str.strip().str.lower()
    target_col = None
    for candidate in ["type", "attack_type", "label"]:
        if candidate in full.columns:
            if candidate == "label" and ("type" in full.columns or "attack_type" in full.columns):
                continue
            target_col = candidate
            break
    if target_col is not None:
        full = full.rename(columns={target_col: "__label__"})
    else:
        label_cols = [c for c in full.columns if "label" in c or "type" in c or "attack" in c]
        full = full.rename(columns={label_cols[0]: "__label__"}) if label_cols else full.assign(__label__="normal")
    y = map_labels(full, dataset_name).to_numpy()
    return full, y

In [ ]:
def _build_feature_frame(full):
    X_df = full.drop(columns=["__label__"], errors="ignore")
    X_df, bucket_labels = _bucketize_ports(X_df)
    dst_bucket = bucket_labels.get("dst_port", pd.Series(["low"] * len(X_df)))
    cat_cols = [c for c in _LOW_CARD_CATS if c in X_df.columns]
    X_df = _encode_categoricals(X_df, cat_cols)
    X_df = X_df.drop(columns=[c for c in X_df.columns if c in _DROP_COLS], errors="ignore")
    return X_df, dst_bucket

In [ ]:
def _clean_and_standardize(X_df):
    for c in X_df.columns:
        if not pd.api.types.is_numeric_dtype(X_df[c]):
            X_df[c] = pd.to_numeric(X_df[c], errors="coerce")
    X_df = X_df.replace([np.inf, -np.inf], np.nan)
    X_df = X_df.dropna(axis=1, how="all").fillna(X_df.median(numeric_only=True)).fillna(0.0)
    X_np = X_df.astype(np.float32).to_numpy()
    mean, std = X_np.mean(axis=0), X_np.std(axis=0)
    std[std == 0] = 1.0
    return (X_np - mean) / std

In [ ]:
def load_dataset(dataset_name, max_rows=MAX_SAMPLES):
    """Loads and harmonizes a dataset. Raises FileNotFoundError if the
    expected CSVs are absent, unless ALLOW_SYNTHETIC_DEBUG_DATA is True."""
    path, csv_files = _resolve_csv_files(dataset_name)
    if not csv_files:
        return _synthetic_debug_data(dataset_name, path, max_rows)
    full, y = _read_csvs_and_extract_label(csv_files, dataset_name)
    X_df, dst_bucket = _build_feature_frame(full)
    X_np = _clean_and_standardize(X_df)
    meta = pd.DataFrame({"dst_port_bucket": dst_bucket.to_numpy()})
    if max_rows and len(X_np) > max_rows:
        idx = np.random.choice(len(X_np), max_rows, replace=False)
        X_np, y, meta = X_np[idx], y[idx], meta.iloc[idx].reset_index(drop=True)
    print(f"Loaded {dataset_name}: X={X_np.shape}, class distribution={np.bincount(y, minlength=5)}")
    return X_np, y, meta

In [ ]:
def dirichlet_partition(y, n_clients, alpha=0.1, seed=42):
    rng = np.random.default_rng(seed)
    client_indices = [[] for _ in range(n_clients)]
    for c in np.unique(y):
        idx_c = np.where(y == c)[0]
        rng.shuffle(idx_c)
        proportions = rng.dirichlet(np.repeat(alpha, n_clients))
        proportions = proportions / proportions.sum()
        splits = (np.cumsum(proportions) * len(idx_c)).astype(int)[:-1]
        for i, split in enumerate(np.split(idx_c, splits)):
            client_indices[i].extend(split.tolist())
    return [np.array(sorted(idx)) for idx in client_indices]

## Models

In [ ]:
class EndpointMLP(nn.Module):
    def __init__(self, input_dim=39, num_classes=5):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 32)
        self.fc2 = nn.Linear(32, 16)
        self.fc3 = nn.Linear(16, num_classes)
        self.act = nn.LeakyReLU()

    def features(self, x):
        """Penultimate-layer representation, used as the embedding space
        for BCES novelty scoring N(x) rather than raw input features."""
        h = self.act(self.fc1(x))
        return self.act(self.fc2(h))

    def forward(self, x):
        return self.fc3(self.features(x))

In [ ]:
class Endpoint1DCNN(nn.Module):
    def __init__(self, input_dim=39, num_classes=5):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.fc = nn.Linear(128 * input_dim, num_classes)

    def forward(self, x):
        x = x.unsqueeze(1)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = x.view(x.size(0), -1)
        return self.fc(x)

## Real INT8 quantization + honest measurement

In [ ]:
def quantize_model_int8(model):
    """Real dynamic INT8 quantization (genuine qint8 weight tensors) for
    Linear layers. torch.quantization doesn't support Conv1d, so
    Endpoint1DCNN is returned unquantized with a printed caveat."""
    model_cpu = model.to("cpu").eval()
    if isinstance(model_cpu, Endpoint1DCNN):
        print("[INFO] Endpoint1DCNN: dynamic INT8 quantization is not supported "
              "for Conv1d by torch.quantization; reporting FP32 footprint for "
              "this architecture and flagging it as not MCU-deployable as-is.")
        return model_cpu, False
    q_model = torch.quantization.quantize_dynamic(model_cpu, {nn.Linear}, dtype=torch.qint8)
    return q_model, True

In [ ]:
def model_size_kb(model):
    """Real serialized size of the state_dict (true int8 buffer sizes for
    a dynamically-quantized model, not a parameter-count formula)."""
    buf = io.BytesIO()
    torch.save(model.state_dict(), buf)
    return buf.getbuffer().nbytes / 1024.0

In [ ]:
def estimate_peak_activation_kb(model, sample_input):
    """Sums each layer's output tensor size during one forward pass. This
    is an UPPER BOUND on peak SRAM activation (a real allocator reuses
    buffers across layers)."""
    sizes = []

    def hook(_module, _inp, out):
        t = out[0] if isinstance(out, tuple) else out
        if torch.is_tensor(t):
            sizes.append(t.element_size() * t.nelement())

    handles = [m.register_forward_hook(hook) for m in model.modules() if len(list(m.children())) == 0]
    with torch.no_grad():
        model(sample_input)
    for h in handles:
        h.remove()
    return sum(sizes) / 1024.0

In [ ]:
def benchmark_latency_ms(model, sample_input, n_runs=300, warmup=30):
    """Measured host-CPU wall-clock latency for a single-sample forward
    pass; a proxy for on-device Cortex-M4 latency, not a hardware
    measurement."""
    model.eval()
    with torch.no_grad():
        for _ in range(warmup):
            model(sample_input)
        times = []
        for _ in range(n_runs):
            t0 = time.perf_counter()
            model(sample_input)
            times.append((time.perf_counter() - t0) * 1000.0)
    return float(np.median(times)), float(np.std(times))

In [ ]:
def profile_endpoint_model(model, input_dim, is_quantized_request):
    sample = torch.randn(1, input_dim)
    if is_quantized_request:
        eval_model, really_quantized = quantize_model_int8(model)
    else:
        eval_model, really_quantized = model.to("cpu").eval(), False
    size_kb = model_size_kb(eval_model)
    act_kb = estimate_peak_activation_kb(eval_model, sample)
    lat_med, lat_std = benchmark_latency_ms(eval_model, sample)
    return {
        "eval_model": eval_model, "really_quantized": really_quantized,
        "flash_kb": size_kb, "peak_activation_kb_upper_bound": act_kb,
        "latency_ms_host_cpu_median": lat_med, "latency_ms_host_cpu_std": lat_std,
    }

## Federated training, DP-SGD, trust-aware aggregation

In [ ]:
def _maybe_wrap_dp(model, optimizer, dataloader, use_dp, accountant):
    if not (use_dp and OPACUS_AVAILABLE):
        return model, optimizer, dataloader, None
    privacy_engine = PrivacyEngine(accountant="rdp")
    if accountant is not None:
        privacy_engine.accountant.load_state_dict(accountant.state_dict())
    model, optimizer, dataloader = privacy_engine.make_private(
        module=model, optimizer=optimizer, data_loader=dataloader,
        noise_multiplier=DP_NOISE_MULTIPLIER, max_grad_norm=DP_CLIPPING_NORM,
    )
    return model, optimizer, dataloader, privacy_engine

In [ ]:
def _run_local_epoch(model, dataloader, optimizer, criterion, use_prox, global_model):
    total_loss, n_batches = 0.0, 0
    for X_batch, y_batch in dataloader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        if use_prox and global_model is not None:
            prox_term = sum((w - w_t).norm(2) for w, w_t in zip(model.parameters(), global_model.parameters()))
            loss = loss + (FEDPROX_MU / 2) * prox_term
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1
    return total_loss / max(n_batches, 1)

In [ ]:
def local_train(model, dataloader, epochs, use_prox=False, global_model=None, use_dp=False,
                 class_weights=None, accountant=None):
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    model, optimizer, dataloader, privacy_engine = _maybe_wrap_dp(model, optimizer, dataloader, use_dp, accountant)

    epoch_bar = tqdm(range(epochs), desc="    local epochs", leave=False)
    for _ in epoch_bar:
        epoch_loss = _run_local_epoch(model, dataloader, optimizer, criterion, use_prox, global_model)
        epoch_bar.set_postfix(loss=f"{epoch_loss:.4f}")

    if privacy_engine is not None and accountant is not None:
        accountant.load_state_dict(privacy_engine.accountant.state_dict())
    return model._module.state_dict() if hasattr(model, "_module") else model.state_dict()

In [ ]:
def evaluate_model(model, dataloader, device=None):
    """device defaults to the model's own parameter device. Quantized
    models only support CPU kernels, so callers must pass device=cpu
    explicitly when evaluating a quantized model."""
    model.eval()
    if device is None:
        try:
            device = next(model.parameters()).device
        except StopIteration:
            device = torch.device("cpu")
    all_preds, all_labels = [], []
    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch = X_batch.to(device)
            preds = model(X_batch).argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(y_batch.numpy())
    acc = float(np.mean(np.array(all_preds) == np.array(all_labels)))
    return acc, all_preds, all_labels

In [ ]:
def _compute_trust_scores(client_weights, client_metrics, client_reliability, rho):
    q_k = np.array([m["val_acc"] for m in client_metrics])
    denom = q_k.max() - q_k.min() or 1e-8
    q_k_norm = (q_k - q_k.min()) / denom
    for i in range(len(client_weights)):
        client_reliability[i] = rho * client_reliability[i] + (1 - rho) * q_k_norm[i]
    r_k = np.array([client_reliability[i] for i in range(len(client_weights))])
    return q_k_norm, r_k

In [ ]:
def _cosine_similarity_scores(client_weights):
    flat = [torch.cat([w.flatten().cpu() for w in wd.values()]) for wd in client_weights]
    stacked = torch.stack(flat)
    median_w = torch.median(stacked, dim=0).values
    s_k = np.array([max(0.0, F.cosine_similarity(stacked[i].unsqueeze(0), median_w.unsqueeze(0)).item())
                    for i in range(len(client_weights))])
    return s_k

In [ ]:
def trust_aware_aggregation(global_model, client_weights, client_metrics, client_reliability, rho=0.5):
    q_k_norm, r_k = _compute_trust_scores(client_weights, client_metrics, client_reliability, rho)
    s_k = _cosine_similarity_scores(client_weights)
    tau_k = q_k_norm + r_k + s_k
    T = 0.5
    alpha_k = np.exp(tau_k / T) / np.sum(np.exp(tau_k / T))
    global_dict = global_model.state_dict()
    for key in global_dict.keys():
        global_dict[key] = sum(client_weights[i][key] * alpha_k[i] for i in range(len(client_weights)))
    global_model.load_state_dict(global_dict)
    return global_model, client_reliability

In [ ]:
def fedavg_aggregation(global_model, client_weights, client_sizes):
    total = sum(client_sizes)
    global_dict = global_model.state_dict()
    for key in global_dict.keys():
        global_dict[key] = sum(client_weights[i][key] * (client_sizes[i] / total) for i in range(len(client_weights)))
    global_model.load_state_dict(global_dict)
    return global_model

## Budget-Constrained Explanation Scheduling (real signals, real XAI timing)

In [ ]:
class NoveltyTracker:
    """Rolling buffer of recently-explained event embeddings, used to
    compute N(x) as a real nearest-neighbor distance."""

    def __init__(self, maxlen=BCES_NOVELTY_BUFFER_SIZE):
        self.buffer = deque(maxlen=maxlen)

    def novelty(self, embedding):
        if len(self.buffer) == 0:
            return 1.0
        emb = embedding / (np.linalg.norm(embedding) + 1e-8)
        sims = [np.dot(emb, b) for b in self.buffer]
        return float(1.0 - max(sims))

    def update(self, embedding):
        norm_emb = embedding / (np.linalg.norm(embedding) + 1e-8)
        self.buffer.append(norm_emb)

In [ ]:
def device_criticality(dst_port_bucket):
    return 1.0 if dst_port_bucket in HIGH_CRITICALITY_PORT_BUCKETS else 0.3


def attack_severity(probs):
    return float(sum(probs[c] * SEVERITY_TABLE.get(c, 0.0) for c in range(len(probs))))

In [ ]:
def _score_one_event(model, x, dst_port_bucket, cost_estimator, idx, novelty_tracker, weights):
    w_u, w_r, w_a, w_n = weights
    logits = model(x.unsqueeze(0))
    probs = F.softmax(logits, dim=1).cpu().numpy().ravel()
    u = 1.0 - probs.max()
    embedding = x.numpy()
    r = device_criticality(dst_port_bucket)
    a = attack_severity(probs)
    n = novelty_tracker.novelty(embedding)
    v = w_u * u + w_r * r + w_a * a + w_n * n
    cost = cost_estimator(idx)
    return {"idx": idx, "v": v, "cost": cost, "rho": v / (cost + 1e-4), "x": x, "embedding": embedding}

In [ ]:
class BCESScheduler:
    def __init__(self, budget_ms, w_u=1.0, w_r=0.5, w_a=1.0, w_n=0.5):
        self.budget = budget_ms
        self.weights = (w_u, w_r, w_a, w_n)
        self.novelty_tracker = NoveltyTracker()

    def score_events(self, events, model, dst_port_buckets, xai_cost_estimator):
        model.eval()
        with torch.no_grad():
            return [_score_one_event(model, x, dst_port_buckets[i], xai_cost_estimator, i,
                                       self.novelty_tracker, self.weights)
                    for i, (x, _y) in enumerate(events)]

    def schedule_greedy(self, scored):
        ordered = sorted(scored, key=lambda e: e["rho"], reverse=True)
        selected, budget_left = [], self.budget
        for e in ordered:
            if e["cost"] <= budget_left:
                selected.append(e)
                budget_left -= e["cost"]
                self.novelty_tracker.update(e["embedding"])
        return selected

In [ ]:
def _knapsack_fill(items, units):
    dp = np.zeros(units + 1)
    keep = [[False] * (units + 1) for _ in items]
    for i, (v, w) in enumerate(items):
        for b in range(units, w - 1, -1):
            if dp[b - w] + v > dp[b]:
                dp[b] = dp[b - w] + v
                keep[i][b] = True
    return dp, keep


def _knapsack_backtrack(items, keep, units):
    b, chosen = units, []
    for i in range(len(items) - 1, -1, -1):
        if keep[i][b]:
            chosen.append(i)
            b -= items[i][1]
    return chosen

In [ ]:
def exact_knapsack(scored, budget_ms, cost_resolution=0.5):
    """Exact 0-1 knapsack via DP over integer-discretized costs; the
    ablation upper bound against which BCES's greedy approximation is
    measured (Section 5.6)."""
    units = int(round(budget_ms / cost_resolution))
    items = [(e["v"], max(1, int(round(e["cost"] / cost_resolution)))) for e in scored]
    dp, keep = _knapsack_fill(items, units)
    chosen = _knapsack_backtrack(items, keep, units)
    return float(dp[units]), [scored[i] for i in chosen]

In [ ]:
def real_xai_explain(x_np, predict_fn, background, method="lime"):
    """Calls a real shap/lime explainer and times it. Raises if the
    requested library is unavailable -- callers must check availability
    before invoking this rather than silently substituting a constant."""
    t0 = time.perf_counter()
    if method == "shap":
        if not SHAP_AVAILABLE:
            raise RuntimeError("shap is not installed")
        explainer = shap.KernelExplainer(predict_fn, background)
        explainer.shap_values(x_np.reshape(1, -1), nsamples=100, silent=True)
    elif method == "lime":
        if not LIME_AVAILABLE:
            raise RuntimeError("lime is not installed")
        explainer = LimeTabularExplainer(background, mode="classification", discretize_continuous=False)
        explainer.explain_instance(x_np, predict_fn, num_features=10, num_samples=200)
    else:
        raise ValueError(method)
    return (time.perf_counter() - t0) * 1000.0

## Metrics

In [ ]:
def evaluate_metrics(y_true, y_pred):
    return {
        "f1_macro": f1_score(y_true, y_pred, average="macro"),
        "precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
    }

## Main multi-seed experiment driver

### Per-seed setup helpers

In [ ]:
def _prepare_seed_data(seed, X, y, dst_port_bucket_all):
    return train_test_split(X, y, dst_port_bucket_all, test_size=0.2, random_state=seed, stratify=y)

In [ ]:
def _build_class_weights(y_train):
    classes_idx = np.unique(y_train)
    weights = np.clip(compute_class_weight("balanced", classes=classes_idx, y=y_train), 0.5, 3.0)
    full_weights = np.ones(5, dtype=np.float32)
    for i, c in enumerate(classes_idx):
        full_weights[c] = weights[i]
    return torch.tensor(full_weights, dtype=torch.float32).to(DEVICE)

In [ ]:
def _build_loaders(X_train, y_train, X_test, y_test, client_indices):
    test_loader = DataLoader(TensorDataset(torch.tensor(X_test), torch.tensor(y_test)), batch_size=BATCH, shuffle=False)
    client_loaders = [
        DataLoader(TensorDataset(torch.tensor(X_train[idx]), torch.tensor(y_train[idx])), batch_size=BATCH, shuffle=True)
        for idx in client_indices if len(idx) > 0
    ]
    return test_loader, client_loaders

### Federated round loop (per-client and per-round progress)

In [ ]:
def _run_clients_for_round(global_model, client_loaders, use_prox, use_dp, use_trust_agg, class_weights, accountant):
    input_dim = global_model.fc1.in_features
    client_weights, client_metrics, client_sizes = [], [], []
    for loader in tqdm(client_loaders, desc="    clients", leave=False):
        client_model = EndpointMLP(input_dim=input_dim).to(DEVICE)
        client_model.load_state_dict(global_model.state_dict())
        w = local_train(client_model, loader, epochs=LOCAL_EPOCHS, use_prox=use_prox, global_model=global_model,
                         use_dp=use_dp, class_weights=class_weights, accountant=accountant)
        client_weights.append(w)
        client_sizes.append(len(loader.dataset))
        if use_trust_agg:
            acc, _, _ = evaluate_model(client_model, loader)
            client_metrics.append({"val_acc": acc})
    return client_weights, client_metrics, client_sizes

In [ ]:
def _aggregate_round(global_model, client_weights, client_metrics, client_sizes, client_reliability, use_trust_agg):
    if use_trust_agg:
        return trust_aware_aggregation(global_model, client_weights, client_metrics, client_reliability)
    return fedavg_aggregation(global_model, client_weights, client_sizes), client_reliability

In [ ]:
def run_federated_rounds(global_model, client_loaders, test_loader, use_prox, use_dp, use_trust_agg,
                          class_weights, accountant, client_reliability):
    round_f1s = []
    for r in tqdm(range(ROUNDS), desc="  rounds", leave=False):
        client_weights, client_metrics, client_sizes = _run_clients_for_round(
            global_model, client_loaders, use_prox, use_dp, use_trust_agg, class_weights, accountant)
        global_model, client_reliability = _aggregate_round(
            global_model, client_weights, client_metrics, client_sizes, client_reliability, use_trust_agg)
        _, preds, labels = evaluate_model(global_model, test_loader)
        round_f1 = f1_score(labels, preds, average="macro")
        round_f1s.append(round_f1)
        tqdm.write(f"    round {r + 1}/{ROUNDS} f1_macro={round_f1:.4f}")
    return global_model, client_reliability, round_f1s

### Topology orchestration (profiling, evaluation, optional BCES)

In [ ]:
def _profile_and_evaluate(global_model, input_dim, use_quant, X_test, y_test):
    profile = profile_endpoint_model(global_model, input_dim, is_quantized_request=use_quant)
    eval_model = profile["eval_model"]  # always CPU-resident; quantized models are CPU-only in PyTorch
    eval_loader = DataLoader(TensorDataset(torch.tensor(X_test), torch.tensor(y_test)), batch_size=BATCH, shuffle=False)
    _, preds, labels = evaluate_model(eval_model, eval_loader, device=torch.device("cpu"))
    metrics = evaluate_metrics(labels, preds)
    metrics.update({
        "flash_kb": profile["flash_kb"],
        "peak_activation_kb_upper_bound": profile["peak_activation_kb_upper_bound"],
        "latency_ms_host_cpu_median": profile["latency_ms_host_cpu_median"],
        "really_quantized": profile["really_quantized"],
    })
    return profile, metrics

In [ ]:
def run_topology(name, client_loaders, test_loader, class_weights, input_dim, X_test, y_test, bucket_test,
                  use_prox, use_quant, use_trust_agg, use_dp, dp_sigma=None, with_bces=False):
    global DP_NOISE_MULTIPLIER
    old_sigma = DP_NOISE_MULTIPLIER
    DP_NOISE_MULTIPLIER = dp_sigma if dp_sigma is not None else DP_NOISE_MULTIPLIER

    global_model = EndpointMLP(input_dim=input_dim).to(DEVICE)
    client_reliability = {i: 0.5 for i in range(len(client_loaders))}
    accountant = RDPAccountant() if (use_dp and OPACUS_AVAILABLE) else None

    global_model, client_reliability, round_f1s = run_federated_rounds(
        global_model, client_loaders, test_loader, use_prox, use_dp, use_trust_agg,
        class_weights, accountant, client_reliability)
    epsilon = accountant.get_epsilon(delta=1e-5) if accountant is not None else None

    profile, metrics = _profile_and_evaluate(global_model, input_dim, use_quant, X_test, y_test)
    metrics["dp_epsilon"] = epsilon
    if with_bces and BCES_USE_REAL_XAI:
        metrics.update(run_bces_block(profile["eval_model"], X_test, y_test, bucket_test))

    DP_NOISE_MULTIPLIER = old_sigma
    return metrics, round_f1s

### Centralized FP32 baseline and topology table

In [ ]:
def _run_centralized_fp32(X_train, y_train, test_loader, class_weights, input_dim):
    model = EndpointMLP(input_dim=input_dim).to(DEVICE)
    train_loader = DataLoader(TensorDataset(torch.tensor(X_train), torch.tensor(y_train)), batch_size=BATCH, shuffle=True)
    local_train(model, train_loader, epochs=EPOCHS, class_weights=class_weights)
    profile = profile_endpoint_model(model, input_dim, is_quantized_request=False)
    eval_model = profile["eval_model"].to(DEVICE)
    _, preds, labels = evaluate_model(eval_model, test_loader)
    metrics = evaluate_metrics(labels, preds)
    metrics.update({k: profile[k] for k in ["flash_kb", "peak_activation_kb_upper_bound", "latency_ms_host_cpu_median"]})
    return metrics, [metrics["f1_macro"]] * ROUNDS

In [ ]:
def _run_named_topology(name, kwargs, seed_results, convergence, client_loaders, test_loader,
                         class_weights, input_dim, X_test, y_test, bucket_test, dp_sigma):
    metrics, round_f1s = run_topology(name, client_loaders, test_loader, class_weights, input_dim,
                                       X_test, y_test, bucket_test, dp_sigma=dp_sigma, **kwargs)
    convergence[name] = round_f1s
    seed_results[name] = metrics
    tqdm.write(f"  [topology done] {name}: {metrics}")

In [ ]:
def _run_all_topologies(seed_results, convergence, client_loaders, test_loader, class_weights,
                         input_dim, X_test, y_test, bucket_test):
    topology_specs = [
        (RUN_FEDAVG_INT8, "FedAvg INT8", dict(use_prox=False, use_quant=True, use_trust_agg=False, use_dp=False)),
        (RUN_FEDPROX_FP32, "FedProx FP32", dict(use_prox=True, use_quant=False, use_trust_agg=False, use_dp=False)),
        (RUN_FEDX_TINYIDS, "FedX-TinyIDS", dict(use_prox=True, use_quant=True, use_trust_agg=True,
                                                 use_dp=OPACUS_AVAILABLE, with_bces=True)),
        (RUN_ABLATION_TRUST, "FedX (No Trust Agg)", dict(use_prox=True, use_quant=True, use_trust_agg=False,
                                                          use_dp=OPACUS_AVAILABLE)),
    ]
    for flag, name, kwargs in topology_specs:
        if flag:
            _run_named_topology(name, kwargs, seed_results, convergence, client_loaders, test_loader,
                                 class_weights, input_dim, X_test, y_test, bucket_test, dp_sigma=None)
    if RUN_ABLATION_DP and OPACUS_AVAILABLE:
        for sigma in DP_SIGMA_SWEEP:
            kwargs = dict(use_prox=True, use_quant=True, use_trust_agg=True, use_dp=True)
            _run_named_topology(f"FedX DP sigma={sigma}", kwargs, seed_results, convergence, client_loaders,
                                 test_loader, class_weights, input_dim, X_test, y_test, bucket_test, dp_sigma=sigma)

In [ ]:
def run_one_seed(seed, X, y, dst_port_bucket_all, input_dim):
    set_seed(seed)
    X_train, X_test, y_train, y_test, bucket_train, bucket_test = _prepare_seed_data(seed, X, y, dst_port_bucket_all)
    class_weights = _build_class_weights(y_train)
    client_indices = dirichlet_partition(y_train, N_CLIENTS, alpha=NON_IID_ALPHA, seed=seed)
    test_loader, client_loaders = _build_loaders(X_train, y_train, X_test, y_test, client_indices)

    seed_results, convergence = {}, {}
    if RUN_CENTRALIZED_FP32:
        seed_results["Centralized FP32"], convergence["Centralized FP32"] = _run_centralized_fp32(
            X_train, y_train, test_loader, class_weights, input_dim)
    _run_all_topologies(seed_results, convergence, client_loaders, test_loader, class_weights,
                         input_dim, X_test, y_test, bucket_test)
    return seed_results, convergence

### BCES ablation block (random / uncertainty-only / exact-knapsack baselines)

In [ ]:
def _select_bces_events(X_test, y_test, bucket_test):
    anomaly_idx = np.where(y_test > 0)[0][:60]  # kept small: real SHAP/LIME calls are expensive
    events = [(torch.tensor(X_test[i]), y_test[i]) for i in anomaly_idx]
    buckets = [bucket_test.iloc[i] if hasattr(bucket_test, "iloc") else bucket_test[i] for i in anomaly_idx]
    background = X_test[np.random.choice(len(X_test), min(50, len(X_test)), replace=False)]
    return events, buckets, background

In [ ]:
def _make_bces_cost_estimator(model, events, background):
    def predict_fn(x_batch):
        with torch.no_grad():
            return F.softmax(model(torch.tensor(x_batch, dtype=torch.float32)), dim=1).cpu().numpy()

    measured_costs = {}

    def cost_estimator(idx):
        if idx in measured_costs:
            return measured_costs[idx]
        cost = real_xai_explain(events[idx][0].numpy(), predict_fn, background, method="lime")
        measured_costs[idx] = cost
        return cost

    return cost_estimator, measured_costs

In [ ]:
def _budgeted_select(scored, order_key_fn, budget_ms, shuffle=False, seed=0):
    """Shared greedy selection used by both the random and uncertainty-only
    ablation baselines (Section 5.6), differing only in event ordering."""
    if shuffle:
        ordered = np.random.default_rng(seed).permutation(scored)
    else:
        ordered = sorted(scored, key=order_key_fn, reverse=True)
    selected, b = [], budget_ms
    for e in ordered:
        if e["cost"] <= b:
            selected.append(e)
            b -= e["cost"]
    return selected

In [ ]:
def _summarize_bces(events, measured_costs, bces_selected, rand_selected, unc_selected, opt_utility):
    bces_utility = sum(e["v"] for e in bces_selected)
    bces_cost = sum(e["cost"] for e in bces_selected)
    return {
        "xai_events_considered": len(events),
        "xai_events_explained_bces": len(bces_selected),
        "xai_total_cost_ms_bces": bces_cost,
        "xai_mean_measured_cost_ms": float(np.mean(list(measured_costs.values()))) if measured_costs else None,
        "bces_utility": bces_utility,
        "random_utility": sum(e["v"] for e in rand_selected),
        "uncertainty_only_utility": sum(e["v"] for e in unc_selected),
        "exact_knapsack_optimal_utility": opt_utility,
        "bces_to_optimal_ratio": (bces_utility / opt_utility) if opt_utility > 0 else None,
    }

In [ ]:
def run_bces_block(model, X_test, y_test, bucket_test):
    """Runs the real BCES scheduler plus the random / uncertainty-only /
    exact-knapsack baselines required by Section 5.6, using real measured
    SHAP/LIME timing as the per-event cost."""
    events, buckets, background = _select_bces_events(X_test, y_test, bucket_test)
    cost_estimator, measured_costs = _make_bces_cost_estimator(model, events, background)
    scheduler = BCESScheduler(budget_ms=BCES_BUDGET_MS)
    scored = scheduler.score_events(events, model, buckets, cost_estimator)

    bces_selected = scheduler.schedule_greedy([dict(e) for e in scored])
    rand_selected = _budgeted_select(scored, None, BCES_BUDGET_MS, shuffle=True)
    unc_selected = _budgeted_select(scored, lambda e: e["v"], BCES_BUDGET_MS)
    opt_utility, _ = exact_knapsack(scored, BCES_BUDGET_MS)
    return _summarize_bces(events, measured_costs, bces_selected, rand_selected, unc_selected, opt_utility)

### Cross-seed summary & significance testing

In [ ]:
def _collect_metric_rows(all_seed_results):
    rows = defaultdict(list)
    for seed_dict in all_seed_results:
        for name, metrics in seed_dict.items():
            rows[name].append(metrics)
    return rows

In [ ]:
def summarize_across_seeds(all_seed_results):
    rows = _collect_metric_rows(all_seed_results)
    summary = {}
    for name, metric_dicts in rows.items():
        keys = {k for d in metric_dicts for k, v in d.items() if isinstance(v, (int, float)) and v is not None}
        summary[name] = {}
        for k in keys:
            vals = [d[k] for d in metric_dicts if d.get(k) is not None]
            if vals:
                summary[name][f"{k}_mean"] = float(np.mean(vals))
                summary[name][f"{k}_std"] = float(np.std(vals))
    return summary

In [ ]:
def significance_vs_baseline(all_seed_results, target="FedX-TinyIDS", baseline="FedAvg INT8", metric="f1_macro"):
    target_vals = [d[target][metric] for d in all_seed_results if target in d and baseline in d]
    base_vals = [d[baseline][metric] for d in all_seed_results if target in d and baseline in d]
    if len(target_vals) < 2:
        return None
    stat, p = wilcoxon(target_vals, base_vals)
    return {"statistic": float(stat), "p_value": float(p), "n_seeds": len(target_vals)}

## Run Experiment

In [ ]:
X, y, meta = load_dataset(SELECTED_DATASETS[0], max_rows=MAX_SAMPLES)
input_dim = X.shape[1]
dst_port_bucket_all = meta["dst_port_bucket"]
all_seed_results, all_convergence = [], []

In [ ]:
for seed in tqdm(SEEDS, desc="seeds"):
    tqdm.write(f"\n========== SEED {seed} ==========")
    seed_results, convergence = run_one_seed(seed, X, y, dst_port_bucket_all, input_dim)
    all_seed_results.append(seed_results)
    all_convergence.append(convergence)

In [ ]:
summary = summarize_across_seeds(all_seed_results)
sig = significance_vs_baseline(all_seed_results)

In [ ]:
if RUN_ABLATION_BCES:
    print("\nBCES ablation utility ratios already collected per-seed under 'FedX-TinyIDS' "
          "(see xai_* / *_utility fields); aggregate them via the summary above.")

print("\n==================== Final Summary (mean +/- std over seeds) ====================")
print(pd.DataFrame(summary).T.to_string())
if sig:
    print(f"\nWilcoxon signed-rank FedX-TinyIDS vs FedAvg INT8 on f1_macro: {sig}")

In [ ]:
out = {
    "config": {
        "dataset": SELECTED_DATASETS[0], "seeds": SEEDS, "rounds": ROUNDS,
        "n_clients": N_CLIENTS, "non_iid_alpha": NON_IID_ALPHA,
        "opacus_available": OPACUS_AVAILABLE, "shap_available": SHAP_AVAILABLE,
        "lime_available": LIME_AVAILABLE,
    },
    "per_seed_results": all_seed_results,
    "summary_mean_std": summary,
    "significance_vs_fedavg": sig,
}
out_path = RESULTS_DIR / f"run_{int(time.time())}.json"
with open(out_path, "w") as f:
    json.dump(out, f, indent=2, default=float)
print(f"\nResults written to {out_path}")